# v10: The Contextual Manifold

## Sparse Geometric Signal Transport -- Architecture v10

**Lineage:** v9 (geometric attention) + contextual manifold via parallel scan

**Key Change**: Replace `nn.Embedding(max_seq_len, manifold_dim)` with a content-dependent manifold coordinate computed via parallel associative scan (Wilson line / SSM identity).

### The Fatal Line (v1-v9)

```python
self.manifold_coords = nn.Embedding(max_seq_len, manifold_dim)
q = self.manifold_coords(positions)  # positions = torch.arange(T)
```

This makes $q_t$ a fixed vector for position $t$, independent of tokens. The "manifold" was a lookup table. Every downstream component -- memory bank routing, Hopfield gradient, gauge transport -- operated on a fiction.

### The v10 Fix

$$q_t = A(x_t) \odot q_{t-1} + B(x_t) \odot \psi(x_t)$$

where $A(x_t)$ is content-dependent retention, $B(x_t)$ is input gating, $\psi(x_t)$ projects to manifold space. This IS the Wilson line accumulation, IS the selective SSM (Mamba), IS parallelizable via associative scan.

### What Changes Downstream

| Component | v9 (positional) | v10 (contextual) |
|---|---|---|
| Memory bank routing | Same atoms for position 5, always | Different atoms based on context |
| Hopfield gradient | Context-blind dominant force | Context-aware dominant force |
| Cross-position forces | Fighting Hopfield | Complementing Hopfield |
| Activation patterns | Position-dependent | Context-dependent fingerprints |

### Key Discovery: Sparse Fiber Bundle Construction

Tokens = **sparse manipulations on the fiber bundle**. Per-subbundle top-k produces quantifiable activation patterns ($\prod_k \binom{d_k}{\text{top-}k_s}$ possible patterns). With contextual manifold, patterns become context-dependent fingerprints.

### Version History

| Version | Key Innovation | Accuracy | BPC |
|---|---|---|---|
| v1-v2 | Positional Wilson line | 5-12% | -- |
| v3 | GRU context gate (sequential) | 35% | -- |
| v4-v5 | Causal conv / attention-gauge | 18-20% | -- |
| v6 | Sparsity as routing | ~20% | -- |
| v7.x | Unified Langevin + deep supervision | 36.5% | -- |
| v8.x | Per-subbundle memory, Shakespeare | 45% | 2.65 |
| v9 | Per-subbundle sparse attention (4th force) | 45% | 2.65 |
| **v10** | **Contextual manifold (parallel scan)** | **?** | **?** |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from typing import Optional, Tuple, Dict, List
from dataclasses import dataclass
import math
import os
import urllib.request
from tqdm.notebook import tqdm

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


## 1. Data

In [2]:
data_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "v8_real_text", "data")
data_path = os.path.join(data_dir, "input.txt")

if not os.path.exists(data_path):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r") as f:
    text = f.read()

def encode(s): return [ord(c) for c in s]
def decode(t): return "".join(chr(x) for x in t if 0 <= x < 256)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Train: {len(train_data):,} | Val: {len(val_data):,} chars")

Train: 1,003,854 | Val: 111,540 chars


## 2. Configuration

In [3]:
@dataclass
class ArchitectureConfig:
    # === v10 CONFIG ===
    # v9 baseline: Val BPC 2.65, Val Acc 45%, Val PPL 6.3.
    # v10.0: Replace nn.Embedding manifold with ContextualManifold (parallel scan).
    #   All other architecture unchanged from v9 (Phase 1: minimal fix).
    #   Hypothesis: context-aware Hopfield gradient breaks the 45% ceiling.

    fiber_dim: int = 256
    n_subbundles: int = 8
    manifold_dim: int = 128

    vocab_size: int = 256
    max_seq_len: int = 128

    atoms_per_subbundle: int = 128
    k_wta_per_subbundle: int = 16

    n_blocks: int = 4
    context_expand: int = 2
    attn_top_k: int = 8

    langevin_steps: int = 6
    langevin_lr: float = 0.1
    beta_init: float = 1.0
    beta_final: float = 10.0

    sparsity_lambda: float = 0.3
    inhibition_gamma: float = 0.1

    learning_rate: float = 3e-4
    dropout: float = 0.1
    batch_size: int = 16
    seq_len: int = 64
    max_steps: int = 10000
    eval_interval: int = 200
    eval_steps: int = 20

    @property
    def subbundle_dim(self):
        assert self.fiber_dim % self.n_subbundles == 0
        return self.fiber_dim // self.n_subbundles

config = ArchitectureConfig()
print(f"Fiber: {config.fiber_dim} = {config.n_subbundles} x {config.subbundle_dim}")
print(f"Manifold: {config.manifold_dim}d (CONTEXTUAL -- parallel scan)")
print(f"Blocks: {config.n_blocks}, Langevin: {config.langevin_steps} steps")
print(f"Attention: top-{config.attn_top_k} per subbundle per step")

Fiber: 256 = 8 x 32
Manifold: 128d (CONTEXTUAL -- parallel scan)
Blocks: 4, Langevin: 6 steps
Attention: top-8 per subbundle per step


In [4]:
def get_batch(split_data, cfg):
    max_start = len(split_data) - cfg.seq_len - 1
    starts = torch.randint(0, max_start, (cfg.batch_size,))
    return torch.stack([split_data[s:s+cfg.seq_len] for s in starts]).to(device)

## 3. The Contextual Manifold (Core v10 Innovation)

The Wilson line accumulation from gauge theory is mathematically identical to a selective state space model recurrence:

$$q_t = A(x_t) \odot q_{t-1} + B(x_t) \odot \psi(x_t)$$

| Wilson Line (Gauge Theory) | SSM (Mamba) | Interpretation |
|---|---|---|
| Connection 1-form $A$ | State transition $A(x_t)$ | Content-dependent dynamics |
| Path-ordered exponential | Parallel associative scan | Efficient parallel computation |
| Holonomy | Hidden state $q_t$ | Accumulated context |
| Gauge curvature $\Omega$ | Path-dependence of $q_t$ | Context sensitivity |

This is the **only structural change** from v9. Everything downstream benefits automatically because the memory bank, Hopfield gradient, and routing all depend on $q_t$.

In [5]:
def linear_scan(gates, inputs):
    """Compute h_t = gates_t * h_{t-1} + inputs_t for all t.

    Sequential implementation of the parallel associative scan.
    The recurrence is the Wilson line accumulation from gauge theory,
    equivalently the selective state space model (Mamba) recurrence.

    For GPU-parallel O(log T) depth version, replace with Blelloch scan
    or CUDA kernels (e.g., mamba_ssm.selective_scan_fn).

    Args:
        gates: (B, T, D) -- multiplicative retention/decay gates
        inputs: (B, T, D) -- additive inputs (B_t * psi_t)

    Returns:
        h: (B, T, D) -- accumulated manifold coordinates
    """
    B, T, D = gates.shape
    h = torch.empty_like(inputs)
    h[:, 0] = inputs[:, 0]
    for t in range(1, T):
        h[:, t] = gates[:, t] * h[:, t-1] + inputs[:, t]
    return h


class ContextualManifold(nn.Module):
    """Wilson line as parallel associative scan.

    Replaces the positional nn.Embedding with a content-dependent
    manifold coordinate computed from the sparse token sequence:

        q_t = A(x_t) * q_{t-1} + B(x_t) * psi(x_t)

    where:
        A(x_t) = sigmoid(proj_A(x_t)) -- retention gate
        B(x_t) = sigmoid(proj_B(x_t)) -- input gate
        psi(x_t) = proj_psi(x_t)      -- token projected to manifold space
    """

    def __init__(self, cfg):
        super().__init__()
        self.A_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.B_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.psi_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.norm = nn.LayerNorm(cfg.manifold_dim)

    def forward(self, x_sparse):
        """Compute contextual manifold coordinates from sparse token embeddings.

        Args:
            x_sparse: (B, T, D) -- sparse token embeddings

        Returns:
            q: (B, T, d_m) -- contextual manifold coordinates encoding x_{0:t}
        """
        A = torch.sigmoid(self.A_proj(x_sparse))    # (B, T, d_m) -- retention
        B = torch.sigmoid(self.B_proj(x_sparse))    # (B, T, d_m) -- input gate
        psi = self.psi_proj(x_sparse)               # (B, T, d_m) -- token -> manifold

        q = linear_scan(A, B * psi)                 # (B, T, d_m) -- accumulated context
        q = self.norm(q)                             # Stabilize manifold coordinates
        return q

print("ContextualManifold defined -- replaces nn.Embedding(max_seq_len, manifold_dim)")

ContextualManifold defined -- replaces nn.Embedding(max_seq_len, manifold_dim)


## 4. Foundation Components

### SparseTokenEmbedding (v10: uses ContextualManifold)

**v10 change**: `manifold_coords = nn.Embedding(max_seq_len, manifold_dim)` is replaced with `ContextualManifold`. The manifold coordinate $q_t$ is now a function of the full token history $x_{0:t}$.

### PerSubbundleMemoryBank (unchanged)

With contextual $q_t$, the router now sees accumulated context -- different histories produce different attractor landscapes $M_{q_t}$.

In [6]:
class SparseTokenEmbedding(nn.Module):
    """Embed tokens as sparse fiber bundle sections + compute contextual manifold coords.

    v10 CHANGE: nn.Embedding(max_seq_len, manifold_dim) -> ContextualManifold.
    The manifold coordinate q_t is now f(x_0, ..., x_t), not Embedding(t).
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        self.topk_per_sub = max(1, cfg.subbundle_dim // 4)

        # v10: CONTEXTUAL MANIFOLD (replaces nn.Embedding positional lookup)
        self.contextual_manifold = ContextualManifold(cfg)

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.embedding(token_ids)

        # Per-subbundle top-k sparsification (Axiom 1: sparse fiber sections)
        chunks = x.chunk(self.cfg.n_subbundles, dim=-1)
        sparse_chunks = []
        for c in chunks:
            _, idx = torch.topk(c.abs(), self.topk_per_sub, dim=-1)
            mask = torch.zeros_like(c)
            mask.scatter_(-1, idx, 1.0)
            sparse_chunks.append(c * mask)
        x_sparse = torch.cat(sparse_chunks, dim=-1)

        # v10: Contextual manifold coordinates from token content
        q = self.contextual_manifold(x_sparse)

        return x_sparse, q


class PerSubbundleMemoryBank(nn.Module):
    """Dynamic memory bank with per-subbundle dictionaries and routing (Axiom 3).

    v10: With contextual q_t, routing is context-aware. Different token
    histories produce different attractor landscapes M_q.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        sd, K, A = cfg.subbundle_dim, cfg.n_subbundles, cfg.atoms_per_subbundle

        self.dictionaries = nn.ParameterList(
            [nn.Parameter(torch.randn(A, sd) * 0.02) for _ in range(K)]
        )
        self.routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.manifold_dim + sd, A), nn.SiLU(), nn.Linear(A, A)
            )
            for _ in range(K)
        ])

    def forward(self, q, x):
        cfg = self.cfg
        sd = cfg.subbundle_dim
        x_chunks = x.split(sd, dim=-1)
        M_list = []

        for chunk, dic, router in zip(x_chunks, self.dictionaries, self.routers):
            D_n = F.normalize(dic, dim=-1)
            logits = router(torch.cat([q, chunk], dim=-1))
            _, idx = torch.topk(logits, cfg.k_wta_per_subbundle, dim=-1)
            M_list.append(D_n[idx])

        return M_list

print("SparseTokenEmbedding (v10) + PerSubbundleMemoryBank defined")

SparseTokenEmbedding (v10) + PerSubbundleMemoryBank defined


## 5. Per-Subbundle Sparse Attention (from v9)

Each of the $K$ subbundles has its own Q, K, V projections. Attention is causal and sparse (top-$k$ most aligned past positions per query). Unchanged from v9.

In [7]:
class PerSubbundleAttention(nn.Module):
    """Sparse causal attention per subbundle channel (v9).

    Each subbundle attends independently with its own QKV projections.
    Top-k sparse: only k most relevant past positions per query.
    """

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg
        sd = cfg.subbundle_dim
        K = cfg.n_subbundles

        self.W_Q = nn.ModuleList([nn.Linear(sd, sd, bias=False) for _ in range(K)])
        self.W_K = nn.ModuleList([nn.Linear(sd, sd, bias=False) for _ in range(K)])
        self.W_V = nn.ModuleList([nn.Linear(sd, sd, bias=False) for _ in range(K)])

        self.scale = math.sqrt(sd)

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        B, T, D = x_seq.shape
        sd = self.cfg.subbundle_dim
        k = self.cfg.attn_top_k

        x_chunks = x_seq.split(sd, dim=-1)
        out_chunks = []

        causal_mask = torch.triu(
            torch.ones(T, T, device=x_seq.device) * float('-inf'), diagonal=1
        )

        for sub_idx, x_sub in enumerate(x_chunks):
            Q = self.W_Q[sub_idx](x_sub)
            K_mat = self.W_K[sub_idx](x_sub)
            V = self.W_V[sub_idx](x_sub)

            scores = torch.bmm(Q, K_mat.transpose(-2, -1)) / self.scale
            scores = scores + causal_mask.unsqueeze(0)

            if k < T:
                topk_vals, topk_idx = torch.topk(scores, k, dim=-1)
                sparse_scores = torch.full_like(scores, float('-inf'))
                sparse_scores.scatter_(-1, topk_idx, topk_vals)
            else:
                sparse_scores = scores

            attn_weights = F.softmax(sparse_scores, dim=-1)
            attn_out_sub = torch.bmm(attn_weights, V)
            out_chunks.append(attn_out_sub)

        return torch.cat(out_chunks, dim=-1)

print("PerSubbundleAttention defined")

PerSubbundleAttention defined


## 6. Score Function with Four Forces

$$\text{score}(x_t, q_t, s) = \underbrace{-\nabla_x E_{q_t}(x_t; M_{q_t})}_{\text{Context-aware settling}} + \underbrace{\alpha_s \cdot f_{\text{conv}}(\mathbf{x})}_{\text{Local smoothing}} + \underbrace{\alpha_a \cdot f_{\text{attn}}(\mathbf{x})}_{\text{Selective retrieval}} + \underbrace{\gamma\, W_{\text{inh}}\, x_t}_{\text{Competition}}$$

**v10 difference**: Force 1 (Hopfield gradient) is now parameterized by context-aware $M_{q_t}$. The dominant force is no longer context-blind. Instrumented with optional `return_forces` for diagnostic Tests 6-7.

In [8]:
class CausalScoreFunction(nn.Module):
    """Four additive forces: Hopfield + causal conv + inhibition + sparse attention.

    v10: Same structure as v9, but Force 1 is now context-aware because M_q
    is parameterized by contextual manifold coordinates.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # Force 2: Causal convolution (smooth local context)
        self.log_decay = nn.Parameter(torch.linspace(-1.0, 2.0, cfg.fiber_dim))
        expanded = cfg.fiber_dim * cfg.context_expand
        self.context_proj = nn.Sequential(
            nn.Linear(cfg.fiber_dim, expanded), nn.SiLU(),
            nn.Dropout(cfg.dropout), nn.Linear(expanded, cfg.fiber_dim),
        )

        # Force 3: Lateral inhibition
        self.W_inh = nn.Parameter(torch.randn(cfg.fiber_dim, cfg.fiber_dim) * 0.01)

        # Force 4: Per-subbundle sparse attention (selective retrieval)
        self.subbundle_attn = PerSubbundleAttention(cfg)
        self.attn_proj = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.Tanh(),
        )

        # Per-step scheduling
        self.step_context_logits = nn.Parameter(
            torch.linspace(1.0, -1.0, cfg.langevin_steps)
        )
        self.step_attn_logits = nn.Parameter(
            torch.linspace(0.5, -0.5, cfg.langevin_steps)
        )

    def _causal_mix(self, x_seq):
        B, T, D = x_seq.shape
        decay = F.softplus(self.log_decay)
        lags = torch.arange(T, device=x_seq.device).float()
        kernel = torch.exp(-decay.unsqueeze(0) * lags.unsqueeze(-1))
        kernel = kernel / (kernel.sum(0, keepdim=True) + 1e-8)
        x_p = F.pad(x_seq, (0, 0, 0, T))
        k_p = F.pad(kernel, (0, 0, 0, T))
        X = torch.fft.fft(x_p, dim=1)
        K = torch.fft.fft(k_p, dim=0).unsqueeze(0)
        return torch.fft.ifft(X * K, dim=1).real[:, :T, :]

    def hopfield_gradient_subbundle(self, x_chunk, M_k, beta):
        sim = torch.bmm(M_k, x_chunk.unsqueeze(-1)).squeeze(-1)
        w = F.softmax(beta * sim, dim=-1)
        return -torch.bmm(w.unsqueeze(1), M_k).squeeze(1)

    def forward(self, x_seq, M_q_all_list, beta, step_idx, return_forces=False):
        B, T, D = x_seq.shape
        sd = self.cfg.subbundle_dim

        # Force 1: Per-subbundle Hopfield gradient (NOW CONTEXT-AWARE in v10)
        x_flat = x_seq.reshape(B * T, D)
        x_chunks = x_flat.split(sd, dim=-1)
        grad_chunks = [
            self.hopfield_gradient_subbundle(xc, mk, beta)
            for xc, mk in zip(x_chunks, M_q_all_list)
        ]
        grad_E = torch.cat(grad_chunks, dim=-1).reshape(B, T, D)

        # Force 2: Causal context (smooth local mixing)
        x_mixed = self._causal_mix(x_seq)
        context_diff = x_mixed - x_seq
        causal_force = self.context_proj(context_diff.reshape(-1, D)).reshape(B, T, D)
        alpha_ctx = torch.sigmoid(self.step_context_logits[step_idx])
        causal_force = alpha_ctx * causal_force

        # Force 3: Lateral inhibition
        inhibition = self.cfg.inhibition_gamma * (x_flat @ self.W_inh.T).reshape(B, T, D)

        # Force 4: Per-subbundle sparse attention (selective retrieval)
        attn_context = self.subbundle_attn(x_seq)
        attn_diff = attn_context - x_seq
        attn_force = self.attn_proj(attn_diff.reshape(-1, D)).reshape(B, T, D)
        alpha_attn = torch.sigmoid(self.step_attn_logits[step_idx])
        attn_force = alpha_attn * attn_force

        total = grad_E + causal_force + inhibition + attn_force

        if return_forces:
            return total, {
                'hopfield': grad_E.detach(),
                'causal': causal_force.detach(),
                'inhibition': inhibition.detach(),
                'attention': attn_force.detach(),
            }
        return total

print("CausalScoreFunction defined (with force tracking)")

CausalScoreFunction defined (with force tracking)


## 7. Langevin Descent, Block, and CLM

Architecture unchanged from v9 except:
- `SparseTokenEmbedding` now uses `ContextualManifold`
- `SpectralGaugeCLM.forward` supports `return_manifold=True` for diagnostics
- `SequenceLangevinDescent` supports `return_forces=True` for force analysis

In [9]:
class SequenceLangevinDescent(nn.Module):
    """Annealed Langevin dynamics on Modern Hopfield energy landscape (Axiom 4)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.score_fn = CausalScoreFunction(cfg)

    def forward(self, x_seq, M_q_all, return_trajectory=False, return_forces=False):
        cfg = self.cfg
        x = x_seq.clone()
        trajectory = [x.detach().clone()] if return_trajectory else None
        force_history = [] if return_forces else None
        betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps)

        for step in range(cfg.langevin_steps):
            beta_t = betas[step].item()

            if return_forces:
                score, forces = self.score_fn(x, M_q_all, beta_t, step, return_forces=True)
                force_history.append(forces)
            else:
                score = self.score_fn(x, M_q_all, beta_t, step)

            noise = math.sqrt(2.0 * cfg.langevin_lr / beta_t) * torch.randn_like(x)
            x = x - cfg.langevin_lr * score + noise

            if return_trajectory:
                trajectory.append(x.detach().clone())

        result = {'settled': x}
        if return_trajectory:
            result['trajectory'] = trajectory
        if return_forces:
            result['force_history'] = force_history
        return result


class DiffusionRoutingBlock(nn.Module):
    """One block: memory bank query -> Langevin settling -> gated residual."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.memory_bank = PerSubbundleMemoryBank(cfg)
        self.seq_langevin = SequenceLangevinDescent(cfg)
        self.norm = nn.LayerNorm(cfg.fiber_dim)
        self.dropout = nn.Dropout(cfg.dropout)
        self.residual_gate = nn.Sequential(
            nn.Linear(cfg.fiber_dim * 2, cfg.fiber_dim), nn.Sigmoid()
        )

    def forward(self, x_seq, q_coords, return_trajectory=False, return_forces=False):
        B, T, D = x_seq.shape
        q_flat = q_coords.reshape(B * T, -1)
        x_flat = x_seq.reshape(B * T, D)
        M_q_all = self.memory_bank(q_flat, x_flat)

        result = self.seq_langevin(
            x_seq, M_q_all,
            return_trajectory=return_trajectory,
            return_forces=return_forces
        )
        x_settled = result['settled']

        gate_in = torch.cat([x_settled, x_seq], dim=-1)
        gate = self.residual_gate(gate_in.reshape(-1, D * 2)).reshape(B, T, D)
        x_out = self.norm(gate * self.dropout(x_settled) + (1 - gate) * x_seq)

        output = {'x': x_out}
        if return_trajectory:
            output['trajectory'] = result['trajectory']
        if return_forces:
            output['force_history'] = result['force_history']
        return output


class SpectralGaugeCLM(nn.Module):
    """v10: Spectral Gauge CLM with Contextual Manifold.

    The ONLY structural change from v9: SparseTokenEmbedding uses
    ContextualManifold instead of nn.Embedding for manifold coordinates.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = SparseTokenEmbedding(cfg)
        self.blocks = nn.ModuleList([DiffusionRoutingBlock(cfg) for _ in range(cfg.n_blocks)])
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )
        self.register_buffer("block_loss_weights", torch.linspace(0.1, 1.0, cfg.n_blocks))

    def forward(self, token_ids, return_manifold=False, return_forces=False):
        B, T = token_ids.shape
        x, q = self.embedding(token_ids)

        intermediate_logits = []
        block_forces = []
        for block in self.blocks:
            result = block(x, q, return_forces=return_forces)
            x = result['x']
            intermediate_logits.append(self.decoder(x)[:, :-1, :])
            if return_forces and 'force_history' in result:
                block_forces.append(result['force_history'])

        info = {
            "mean_sparsity": (x.abs() < 1e-3).float().mean().item(),
            "intermediate_logits": intermediate_logits,
            "block_loss_weights": self.block_loss_weights,
        }

        if return_manifold:
            info["manifold_coords"] = q
            info["final_repr"] = x

        if return_forces:
            info["block_forces"] = block_forces

        return intermediate_logits[-1], info


model = SpectralGaugeCLM(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_manifold_params = sum(p.numel() for p in model.embedding.contextual_manifold.parameters())
n_attn_params = sum(
    sum(p.numel() for p in blk.seq_langevin.score_fn.subbundle_attn.parameters())
    for blk in model.blocks
)
print(f"Total parameters: {n_params:,}")
print(f"Contextual manifold parameters: {n_manifold_params:,} ({100*n_manifold_params/n_params:.1f}%)")
print(f"Attention parameters: {n_attn_params:,} ({100*n_attn_params/n_params:.1f}%)")
print(f"\n{config.n_blocks} blocks x {config.langevin_steps} Langevin steps x {config.n_subbundles} subbundles")

Total parameters: 3,818,672
Contextual manifold parameters: 98,944 (2.6%)
Attention parameters: 98,304 (2.6%)

4 blocks x 6 Langevin steps x 8 subbundles


## 8. Training

In [ ]:
@torch.no_grad()
def estimate_loss(model, cfg):
    model.eval()
    results = {}
    for name, sd in [("train", train_data), ("val", val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        bces = [0.] * cfg.n_blocks
        for _ in range(cfg.eval_steps):
            b = get_batch(sd, cfg)
            logits, info = model(b)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            tot_sp += info["mean_sparsity"]
            for i, bl in enumerate(info["intermediate_logits"]):
                bces[i] += F.cross_entropy(bl.reshape(-1, cfg.vocab_size), tgt.reshape(-1)).item()
        n = cfg.eval_steps
        results[name] = {"ce": tot_ce/n, "acc": tot_ok/tot_n, "sparsity": tot_sp/n,
                         "block_ces": [c/n for c in bces]}
    model.train()
    return results


optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

history = {
    "step": [], "train_ce": [], "val_ce": [], "train_acc": [], "val_acc": [],
    "train_sparsity": [], "val_sparsity": [],
    "train_bpc": [], "val_bpc": [],
    "block_val_ces": [[] for _ in range(config.n_blocks)], "lr": [],
}

print(f"Training v10 (contextual manifold) on Tiny Shakespeare")
print(f"Steps: {config.max_steps}, Batch: {config.batch_size}, Seq: {config.seq_len}")
print(f"v9 baseline to beat: Val BPC 2.65 | Val Acc 45%")
print("=" * 70)

model.train()
pbar = tqdm(range(config.max_steps), desc="Training", unit="step")

for step in pbar:
    if step % config.eval_interval == 0:
        res = estimate_loss(model, config)
        tr, vl = res["train"], res["val"]
        history["step"].append(step)
        for k in ["ce", "acc", "sparsity"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])
        history["train_bpc"].append(tr["ce"] / math.log(2))
        history["val_bpc"].append(vl["ce"] / math.log(2))
        history["lr"].append(optimizer.param_groups[0]["lr"])
        for i in range(config.n_blocks):
            history["block_val_ces"][i].append(vl["block_ces"][i])
        tqdm.write(
            f"Step {step:5d} | Train CE: {tr['ce']:.3f} | Val CE: {vl['ce']:.3f} | "
            f"Val BPC: {vl['ce']/math.log(2):.2f} | Val PPL: {math.exp(vl['ce']):.1f} | "
            f"Val Acc: {vl['acc']:.1%} | Sp: {vl['sparsity']:.1%}"
        )

    batch = get_batch(train_data, config)
    optimizer.zero_grad()
    logits, info = model(batch)
    targets = batch[:, 1:]

    # Deep supervision: weighted CE from all blocks
    ce_loss = sum(
        w * F.cross_entropy(bl.reshape(-1, config.vocab_size), targets.reshape(-1))
        for bl, w in zip(info["intermediate_logits"], info["block_loss_weights"])
    ) / info["block_loss_weights"].sum()

    # Dictionary coherence regularizer
    dcl = 0.
    nd = 0
    for blk in model.blocks:
        for d in blk.memory_bank.dictionaries:
            Dn = F.normalize(d, dim=-1)
            g = Dn @ Dn.T
            dcl += (g - torch.eye(g.size(0), device=g.device)).pow(2).mean()
            nd += 1
    dcl /= max(nd, 1)

    loss = ce_loss + 0.1 * dcl
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

# Final eval
res = estimate_loss(model, config)
tr, vl = res["train"], res["val"]
history["step"].append(config.max_steps)
for k in ["ce", "acc", "sparsity"]:
    history[f"train_{k}"].append(tr[k])
    history[f"val_{k}"].append(vl[k])
history["train_bpc"].append(tr["ce"] / math.log(2))
history["val_bpc"].append(vl["ce"] / math.log(2))
history["lr"].append(optimizer.param_groups[0]["lr"])
for i in range(config.n_blocks):
    history["block_val_ces"][i].append(vl["block_ces"][i])

print("=" * 70)
print(f"FINAL | Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.2f} | "
      f"Val Acc: {vl['acc']:.1%} | Val PPL: {math.exp(vl['ce']):.1f}")
print(f"v9 baseline: Val BPC 2.65 | Val Acc 45% | Val PPL 6.3")

Training v10 (contextual manifold) on Tiny Shakespeare
Steps: 10000, Batch: 16, Seq: 64
v9 baseline to beat: Val BPC 2.65 | Val Acc 45%


Training:   0%|          | 0/10000 [00:00<?, ?step/s]

Step     0 | Train CE: 5.541 | Val CE: 5.544 | Val BPC: 8.00 | Val PPL: 255.6 | Val Acc: 0.7% | Sp: 0.1%
Step   200 | Train CE: 2.371 | Val CE: 2.381 | Val BPC: 3.43 | Val PPL: 10.8 | Val Acc: 30.7% | Sp: 0.1%
Step   400 | Train CE: 2.198 | Val CE: 2.212 | Val BPC: 3.19 | Val PPL: 9.1 | Val Acc: 35.3% | Sp: 0.1%
Step   600 | Train CE: 2.106 | Val CE: 2.115 | Val BPC: 3.05 | Val PPL: 8.3 | Val Acc: 37.3% | Sp: 0.2%


## 9. Training Diagnostics

In [ ]:
steps = history["step"]
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

axes[0,0].plot(steps, history["train_ce"], 'b-', label='Train')
axes[0,0].plot(steps, history["val_ce"], 'r-', label='Val')
axes[0,0].set_title('Cross-Entropy'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(steps, history["train_bpc"], 'b-', label='Train')
axes[0,1].plot(steps, history["val_bpc"], 'r-', label='Val')
axes[0,1].axhline(y=2.65, color='gray', linestyle='--', alpha=0.7, label='v9 baseline')
axes[0,1].set_title('Bits per Character'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

vp = [math.exp(c) for c in history["val_ce"]]
tp = [math.exp(c) for c in history["train_ce"]]
axes[0,2].plot(steps, tp, 'b-', label='Train'); axes[0,2].plot(steps, vp, 'r-', label='Val')
axes[0,2].set_title('Perplexity'); axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(steps, history["train_acc"], 'b-', label='Train')
axes[1,0].plot(steps, history["val_acc"], 'r-', label='Val')
axes[1,0].axhline(y=0.45, color='gray', linestyle='--', alpha=0.7, label='v9 ceiling')
axes[1,0].set_title('Accuracy'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

gap = [v-t for v,t in zip(history["val_ce"], history["train_ce"])]
axes[1,1].plot(steps, gap, 'purple', lw=2)
axes[1,1].set_title('Generalization Gap (Val - Train CE)'); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(steps, history["val_sparsity"], 'r-')
axes[1,2].set_title('Output Sparsity'); axes[1,2].grid(True, alpha=0.3)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, config.n_blocks))
for i in range(config.n_blocks):
    axes[2,0].plot(steps, history["block_val_ces"][i], color=colors[i], label=f'Block {i}')
axes[2,0].set_title('Per-Block Val CE (deep supervision)'); axes[2,0].legend(); axes[2,0].grid(True, alpha=0.3)

axes[2,1].plot(steps, history["lr"], 'g-', lw=2)
axes[2,1].set_title('Learning Rate'); axes[2,1].grid(True, alpha=0.3)

with torch.no_grad():
    sf = model.blocks[0].seq_langevin.score_fn
    ctx_a = torch.sigmoid(sf.step_context_logits).cpu().numpy()
    attn_a = torch.sigmoid(sf.step_attn_logits).cpu().numpy()
axes[2,2].plot(ctx_a, 'b-o', ms=4, label='Causal conv')
axes[2,2].plot(attn_a, 'r-o', ms=4, label='Attention')
axes[2,2].set_xlabel('Langevin Step'); axes[2,2].set_ylabel('Force Strength')
axes[2,2].set_title('Learned Force Schedule (Block 0)')
axes[2,2].legend(); axes[2,2].grid(True, alpha=0.3); axes[2,2].set_ylim(0, 1)

plt.suptitle('v10: Contextual Manifold -- Training Diagnostics', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nFinal: Val BPC {history['val_bpc'][-1]:.2f} | Val Acc {history['val_acc'][-1]:.1%}")
print(f"v9:    Val BPC 2.65 | Val Acc 45%")
delta_bpc = history['val_bpc'][-1] - 2.65
delta_acc = (history['val_acc'][-1] - 0.45) * 100
print(f"Delta: BPC {delta_bpc:+.2f} | Acc {delta_acc:+.1f}pp")

## 10. Text Generation

In [ ]:
@torch.no_grad()
def generate_text(model, prompt_str, max_new=200, temperature=0.8):
    model.eval()
    cfg = model.cfg
    ids = torch.tensor(encode(prompt_str), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new):
        ctx = ids[:, -cfg.max_seq_len:] if ids.size(1) >= cfg.max_seq_len else ids
        logits, _ = model(ctx)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        ids = torch.cat([ids, torch.multinomial(probs, 1)], dim=1)
    return decode(ids[0].tolist())


print("=" * 60)
print("TEXT GENERATION (v10, temperature=0.8)")
print("=" * 60)
for p in ["ROMEO:\n", "To be or not to ", "The king ", "O, "]:
    print(f"\n--- Prompt: {repr(p)} ---")
    print(generate_text(model, p))

## 11. PhD Diagnostic Suite

Ten tests to validate the contextual manifold hypothesis and characterize the sparse fiber bundle construction.

### Manifold Contextuality (Tests 1-3)
- **Test 1**: Same position, different contexts: $\|q_t^{(1)} - q_t^{(2)}\|$ should be nonzero
- **Test 2**: Manifold trajectory PCA: curved path reflecting content
- **Test 3**: Memory bank atom divergence: Jaccard similarity < 1.0

### Context Accumulation (Tests 4-5)
- **Test 4**: Linear probes on $q_t$: can we decode history from manifold coordinates?
- **Test 5**: Ablation: replace contextual $q_t$ with positional at test time

### Force Cooperation (Tests 6-7)
- **Test 6**: Individual force magnitudes: balanced in v10, Hopfield-dominated in v9
- **Test 7**: Force cosine similarity: positive in v10 (aligned), zero in v9 (orthogonal)

### Gauge Curvature (Test 8)
- **Test 8**: Small-loop holonomy: nonzero = path-dependent transport

### Sparse Fiber Bundle Patterns (Tests 9-10)
- **Test 9**: Activation pattern divergence under context variation
- **Test 10**: Subbundle activation entropy: combinatorial capacity utilization

In [ ]:
@torch.no_grad()
def test_manifold_contextuality(model, cfg):
    """Tests 1-3: Is the manifold actually contextual?"""
    model.eval()

    text_a = "ROMEO:\nO, she doth teach the torches to burn bright!"
    text_b = "JULIET:\nWhat's in a name? That which we call a rose"

    T = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)

    _, info_a = model(ids_a, return_manifold=True)
    _, info_b = model(ids_b, return_manifold=True)

    q_a = info_a["manifold_coords"][0].cpu()
    q_b = info_b["manifold_coords"][0].cpu()

    # Test 1: Position-wise manifold divergence
    divergence = (q_a - q_b).norm(dim=-1).numpy()

    # Test 2: PCA visualization (torch SVD)
    combined = torch.cat([q_a, q_b], dim=0)
    centered = combined - combined.mean(dim=0, keepdim=True)
    U, S, Vh = torch.linalg.svd(centered, full_matrices=False)
    explained = (S[:3] ** 2) / (S ** 2).sum()
    proj = (centered @ Vh.T[:, :3]).numpy()
    q_a_3d = proj[:T]
    q_b_3d = proj[T:]

    # Test 3: Memory bank atom overlap (Jaccard)
    x_a, q_a_t = model.embedding(ids_a)
    x_b, q_b_t = model.embedding(ids_b)
    block = model.blocks[0]

    jaccards = []
    for t in range(T):
        sub_jacs = []
        for k, router in enumerate(block.memory_bank.routers):
            sd = cfg.subbundle_dim
            chunk_a = x_a[0, t:t+1, k*sd:(k+1)*sd]
            chunk_b = x_b[0, t:t+1, k*sd:(k+1)*sd]
            q_at = q_a_t[0, t:t+1]
            q_bt = q_b_t[0, t:t+1]

            logits_a = router(torch.cat([q_at, chunk_a], dim=-1))
            logits_b = router(torch.cat([q_bt, chunk_b], dim=-1))

            _, idx_a = torch.topk(logits_a, cfg.k_wta_per_subbundle, dim=-1)
            _, idx_b = torch.topk(logits_b, cfg.k_wta_per_subbundle, dim=-1)

            set_a = set(idx_a[0].cpu().numpy().tolist())
            set_b = set(idx_b[0].cpu().numpy().tolist())
            jac = len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 1.0
            sub_jacs.append(jac)
        jaccards.append(np.mean(sub_jacs))

    # Plot
    fig = plt.figure(figsize=(18, 5))

    ax1 = fig.add_subplot(131)
    ax1.plot(divergence, 'b-', lw=2)
    ax1.set_xlabel('Position'); ax1.set_ylabel('||q_a - q_b||')
    ax1.set_title('Test 1: Manifold Divergence\n(same position, different contexts)')
    ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='v1-v9 value (0.0)')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(132, projection='3d')
    ax2.plot(*q_a_3d.T, 'b-o', ms=3, label='ROMEO...')
    ax2.plot(*q_b_3d.T, 'r-o', ms=3, label='JULIET...')
    ax2.set_title(f'Test 2: Manifold Trajectories (PCA)\n{explained.sum():.0%} variance in 3D')
    ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(133)
    ax3.plot(jaccards, 'g-', lw=2)
    ax3.set_xlabel('Position'); ax3.set_ylabel('Jaccard Similarity')
    ax3.set_title('Test 3: Memory Bank Atom Overlap\n(1.0 = identical routing)')
    ax3.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='v1-v9 value (1.0)')
    ax3.set_ylim(0, 1.1); ax3.legend(); ax3.grid(True, alpha=0.3)

    plt.suptitle('PhD Diagnostics: Manifold Contextuality (Tests 1-3)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nTest 1: Mean divergence = {divergence.mean():.4f} (v1-v9: 0.0000)")
    print(f"Test 3: Mean Jaccard   = {np.mean(jaccards):.4f} (v1-v9: 1.0000)")
    verdict = "PASS" if divergence.mean() > 0.01 else "FAIL"
    print(f"\nVerdict: {verdict}")

test_manifold_contextuality(model, config)

In [ ]:
@torch.no_grad()
def test_probing_and_ablation(model, cfg):
    """Test 4: Linear probes on q_t. Test 5: Manifold ablation."""
    model.eval()

    # Test 4: Probe q_t to predict last character seen
    n_samples = 2000
    qs, targets = [], []
    for _ in range(n_samples // (cfg.batch_size * cfg.seq_len) + 1):
        batch = get_batch(val_data, cfg)
        _, info = model(batch, return_manifold=True)
        q = info["manifold_coords"][:, 1:, :]
        tgt = batch[:, :-1]
        qs.append(q.reshape(-1, cfg.manifold_dim).cpu())
        targets.append(tgt.reshape(-1).cpu())

    Q = torch.cat(qs, dim=0)[:n_samples]
    Y = torch.cat(targets, dim=0)[:n_samples]

    # Focus on most frequent characters
    counts = torch.bincount(Y, minlength=256)
    top_chars = counts.topk(30).indices
    mask = torch.zeros(len(Y), dtype=torch.bool)
    for c in top_chars:
        mask |= (Y == c)
    Q_sub = Q[mask]
    Y_sub = Y[mask]

    n_train = int(0.8 * len(Q_sub))
    Q_train, Q_test = Q_sub[:n_train], Q_sub[n_train:]
    Y_train, Y_test = Y_sub[:n_train], Y_sub[n_train:]

    Y_onehot = F.one_hot(Y_train.long(), num_classes=256).float()
    W_probe = torch.linalg.lstsq(Q_train, Y_onehot).solution
    preds = (Q_test @ W_probe).argmax(dim=-1)
    probe_acc = (preds == Y_test).float().mean().item()
    baseline_acc = torch.bincount(Y_test.long(), minlength=256).max().item() / len(Y_test)

    # Test 5: Ablation -- contextual vs positional manifold
    contextual_acc, n_eval = 0., 0
    for _ in range(cfg.eval_steps):
        b = get_batch(val_data, cfg)
        logits, _ = model(b)
        tgt = b[:, 1:]
        contextual_acc += (logits.argmax(-1) == tgt).sum().item()
        n_eval += tgt.numel()
    contextual_acc /= n_eval

    # Ablated: override manifold with random positional embedding
    pos_embed = nn.Embedding(cfg.max_seq_len, cfg.manifold_dim).to(device)
    original_forward = model.embedding.contextual_manifold.forward

    def positional_override(x_sparse):
        B, T, D = x_sparse.shape
        pos = torch.arange(T, device=x_sparse.device)
        return pos_embed(pos).unsqueeze(0).expand(B, -1, -1)

    model.embedding.contextual_manifold.forward = positional_override
    positional_acc, n_eval = 0., 0
    for _ in range(cfg.eval_steps):
        b = get_batch(val_data, cfg)
        logits, _ = model(b)
        tgt = b[:, 1:]
        positional_acc += (logits.argmax(-1) == tgt).sum().item()
        n_eval += tgt.numel()
    positional_acc /= n_eval
    model.embedding.contextual_manifold.forward = original_forward

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    bars = axes[0].bar(['Probe\n(from q_t)', 'Baseline\n(majority)'],
                       [probe_acc, baseline_acc], color=['steelblue', 'lightcoral'])
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Test 4: Linear Probe on q_t\n(predict previous character)')
    axes[0].set_ylim(0, 1)
    for bar, val in zip(bars, [probe_acc, baseline_acc]):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.1%}', ha='center', fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')

    bars = axes[1].bar(['Contextual\n(v10)', 'Positional\n(ablated)'],
                       [contextual_acc, positional_acc], color=['steelblue', 'lightcoral'])
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Test 5: Manifold Ablation\n(contextual vs positional at test time)')
    axes[1].set_ylim(0, max(contextual_acc, positional_acc) * 1.3 + 0.01)
    for bar, val in zip(bars, [contextual_acc, positional_acc]):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.1%}', ha='center', fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('PhD Diagnostics: Context Accumulation (Tests 4-5)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nTest 4: Probe accuracy = {probe_acc:.1%} (baseline: {baseline_acc:.1%})")
    print(f"Test 5: Contextual = {contextual_acc:.1%} | Positional (ablated) = {positional_acc:.1%}")
    drop = contextual_acc - positional_acc
    print(f"         Drop = {drop:.1%}pp")

test_probing_and_ablation(model, config)

In [ ]:
@torch.no_grad()
def test_force_analysis(model, cfg):
    """Tests 6-7: Are the forces cooperating?"""
    model.eval()

    batch = get_batch(val_data, cfg)
    _, info = model(batch, return_forces=True)

    all_mags = {'hopfield': [], 'causal': [], 'inhibition': [], 'attention': []}
    all_cosines = {'hopfield_causal': [], 'hopfield_attention': []}

    for block_forces in info["block_forces"]:
        for step_forces in block_forces:
            for name in all_mags:
                f = step_forces[name]
                all_mags[name].append(f.norm(dim=-1).mean().item())

            h = step_forces['hopfield'].reshape(-1, cfg.fiber_dim)
            c = step_forces['causal'].reshape(-1, cfg.fiber_dim)
            a = step_forces['attention'].reshape(-1, cfg.fiber_dim)

            cos_hc = F.cosine_similarity(h, c, dim=-1).mean().item()
            cos_ha = F.cosine_similarity(h, a, dim=-1).mean().item()
            all_cosines['hopfield_causal'].append(cos_hc)
            all_cosines['hopfield_attention'].append(cos_ha)

    n_total = cfg.n_blocks * cfg.langevin_steps

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Test 6: Force magnitudes
    x_pos = np.arange(n_total)
    width = 0.2
    colors_f = {'hopfield': 'steelblue', 'causal': 'coral', 'inhibition': 'seagreen', 'attention': 'orchid'}
    for i, (name, mags) in enumerate(all_mags.items()):
        axes[0].bar(x_pos + i * width, mags, width, label=name, color=colors_f[name], alpha=0.8)
    for b in range(1, cfg.n_blocks):
        axes[0].axvline(x=b * cfg.langevin_steps - 0.5, color='gray', linestyle='--', alpha=0.3)
    axes[0].set_xlabel('Block x Step')
    axes[0].set_ylabel('Mean ||force||')
    axes[0].set_title('Test 6: Force Magnitudes')
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3, axis='y')

    mean_hop = np.mean(all_mags['hopfield'])
    mean_others = np.mean([np.mean(all_mags[k]) for k in ['causal', 'attention', 'inhibition']])
    ratio = mean_hop / (mean_others + 1e-8)

    # Test 7: Cosine similarity
    axes[1].plot(all_cosines['hopfield_causal'], 'b-o', ms=4, label='Hopfield vs Causal')
    axes[1].plot(all_cosines['hopfield_attention'], 'r-o', ms=4, label='Hopfield vs Attention')
    axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    for b in range(1, cfg.n_blocks):
        axes[1].axvline(x=b * cfg.langevin_steps - 0.5, color='gray', linestyle='--', alpha=0.3)
    axes[1].set_xlabel('Block x Step')
    axes[1].set_ylabel('Cosine Similarity')
    axes[1].set_title('Test 7: Force Alignment')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle('PhD Diagnostics: Force Cooperation (Tests 6-7)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    mean_cos_ha = np.mean(all_cosines['hopfield_attention'])
    print(f"\nTest 6: Hopfield/others magnitude ratio = {ratio:.1f}x")
    print(f"Test 7: Mean cos(Hopfield, Attention) = {mean_cos_ha:.3f}")

test_force_analysis(model, config)

In [ ]:
@torch.no_grad()
def test_holonomy(model, cfg):
    """Test 8: Small-loop holonomy -- nonzero curvature = path-dependent transport."""
    model.eval()

    text = "ROMEO:\nBut, soft! What light through yonder window breaks?"
    T = min(cfg.seq_len, len(text))
    ids_orig = torch.tensor(encode(text[:T]), dtype=torch.long).unsqueeze(0).to(device)

    holonomies = []
    swap_positions = range(2, T - 2)

    for pos in swap_positions:
        ids_swap = ids_orig.clone()
        ids_swap[0, pos], ids_swap[0, pos + 1] = ids_swap[0, pos + 1].item(), ids_swap[0, pos].item()

        _, info_orig = model(ids_orig, return_manifold=True)
        _, info_swap = model(ids_swap, return_manifold=True)

        q_orig = info_orig["manifold_coords"][0]
        q_swap = info_swap["manifold_coords"][0]

        downstream_div = (q_orig[pos+2:] - q_swap[pos+2:]).norm(dim=-1).mean().item()
        holonomies.append(downstream_div)

    holonomies = np.array(holonomies)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(list(swap_positions), holonomies, 'b-o', ms=4)
    axes[0].set_xlabel('Swap Position')
    axes[0].set_ylabel('Mean Downstream ||dq||')
    axes[0].set_title('Test 8: Small-Loop Holonomy\n(swap 2 adjacent tokens, measure downstream divergence)')
    axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Zero curvature (v1-v9)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Divergence profile for one example
    mid = T // 2
    ids_swap = ids_orig.clone()
    ids_swap[0, mid], ids_swap[0, mid+1] = ids_swap[0, mid+1].item(), ids_swap[0, mid].item()
    _, info_orig = model(ids_orig, return_manifold=True)
    _, info_swap = model(ids_swap, return_manifold=True)
    q_orig = info_orig["manifold_coords"][0].cpu()
    q_swap = info_swap["manifold_coords"][0].cpu()
    pos_div = (q_orig - q_swap).norm(dim=-1).numpy()

    axes[1].plot(pos_div, 'purple', lw=2)
    axes[1].axvline(x=mid, color='red', linestyle='--', alpha=0.7, label=f'Swap at pos {mid}')
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('||q_orig - q_swap||')
    axes[1].set_title(f'Divergence Profile (swap at position {mid})')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle('PhD Diagnostics: Gauge Curvature (Test 8)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nTest 8: Mean holonomy = {holonomies.mean():.4f}")
    print(f"         Max holonomy = {holonomies.max():.4f}")
    verdict = "PASS" if holonomies.mean() > 0.01 else "FAIL"
    print(f"         {verdict} -- gauge curvature is {'nonzero (path-dependent!)' if verdict == 'PASS' else 'zero (flat geometry)'}")

test_holonomy(model, config)

In [ ]:
@torch.no_grad()
def test_sparse_patterns(model, cfg):
    """Tests 9-10: Sparse fiber bundle activation patterns -- the key discovery."""
    model.eval()

    # Test 9: Activation pattern divergence under context variation
    text_a = "The quality of mercy is not strained, it droppeth as"
    text_b = "The quantity of arrows is not counted, it striketh at"
    T = min(cfg.seq_len, len(text_a), len(text_b))

    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)

    _, info_a = model(ids_a, return_manifold=True)
    _, info_b = model(ids_b, return_manifold=True)
    repr_a = info_a["final_repr"][0].cpu()
    repr_b = info_b["final_repr"][0].cpu()

    sd = cfg.subbundle_dim
    hamming_repr = []
    emd_repr = []

    for t in range(T):
        sub_hammings = []
        sub_emds = []
        for k in range(cfg.n_subbundles):
            ra_sub = repr_a[t, k*sd:(k+1)*sd]
            rb_sub = repr_b[t, k*sd:(k+1)*sd]

            ma = (ra_sub.abs() > ra_sub.abs().mean() * 0.1).float()
            mb = (rb_sub.abs() > rb_sub.abs().mean() * 0.1).float()
            sub_hammings.append((ma - mb).abs().sum().item() / sd)

            sorted_a = ra_sub.abs().sort(descending=True).values
            sorted_b = rb_sub.abs().sort(descending=True).values
            sub_emds.append((sorted_a - sorted_b).abs().sum().item())

        hamming_repr.append(np.mean(sub_hammings))
        emd_repr.append(np.mean(sub_emds))

    # Test 10: Activation pattern entropy across subbundles
    batch = get_batch(val_data, cfg)
    _, batch_info = model(batch, return_manifold=True)
    batch_repr = batch_info["final_repr"].cpu()

    from collections import Counter
    from math import comb

    subbundle_entropies = []
    for k in range(cfg.n_subbundles):
        sub_repr = batch_repr[:, :, k*sd:(k+1)*sd]
        topk = min(4, sd)
        _, top_dims = sub_repr.abs().topk(topk, dim=-1)
        patterns = top_dims.reshape(-1, topk).cpu().numpy()
        pattern_strs = [tuple(sorted(p)) for p in patterns]
        counts = Counter(pattern_strs)
        total = sum(counts.values())
        probs = np.array(list(counts.values())) / total
        entropy = -np.sum(probs * np.log(probs + 1e-10))
        subbundle_entropies.append(entropy)

    theoretical_max = np.log(comb(sd, min(4, sd)))

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(hamming_repr, 'r-', lw=2)
    axes[0].set_xlabel('Position')
    axes[0].set_ylabel('Normalized Hamming Distance')
    axes[0].set_title('Test 9a: Support Mask Divergence\n(same texts, different contexts)')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(emd_repr, 'purple', lw=2)
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('EMD (sorted activations)')
    axes[1].set_title('Test 9b: Activation Pattern Distance')
    axes[1].grid(True, alpha=0.3)

    axes[2].bar(range(cfg.n_subbundles), subbundle_entropies, color='teal', alpha=0.8)
    axes[2].axhline(y=theoretical_max, color='red', linestyle='--', alpha=0.5,
                     label=f'Max entropy = {theoretical_max:.1f}')
    axes[2].set_xlabel('Subbundle Index')
    axes[2].set_ylabel('Pattern Entropy (nats)')
    axes[2].set_title('Test 10: Activation Pattern Entropy')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle('PhD Diagnostics: Sparse Fiber Bundle Patterns (Tests 9-10)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nTest 9: Mean Hamming = {np.mean(hamming_repr):.4f}")
    print(f"         Mean EMD = {np.mean(emd_repr):.4f}")
    print(f"Test 10: Mean subbundle entropy = {np.mean(subbundle_entropies):.2f} nats")
    print(f"          Theoretical max = {theoretical_max:.2f} nats")
    print(f"          Utilization = {np.mean(subbundle_entropies)/theoretical_max:.0%}")

test_sparse_patterns(model, config)

## Summary

### The Single Change
```python
# v1-v9 (BROKEN):
self.manifold_coords = nn.Embedding(max_seq_len, manifold_dim)
q = self.manifold_coords(positions)

# v10 (FIXED):
self.contextual_manifold = ContextualManifold(cfg)
q = self.contextual_manifold(x_sparse)  # q_t = A(x_t) * q_{t-1} + B(x_t) * psi(x_t)
```

### PhD Diagnostic Suite Results

| Test | What It Measures | v9 Expected | v10 Target |
|---|---|---|---|
| 1 | Manifold divergence (same pos, diff context) | 0.0 | >> 0 |
| 2 | Manifold trajectory shape (PCA) | Monotone | Curved, content-dependent |
| 3 | Memory bank atom overlap (Jaccard) | 1.0 | < 1.0 |
| 4 | Linear probe accuracy on q_t | Baseline | >> baseline |
| 5 | Ablation accuracy drop | 0 | Significant |
| 6 | Force magnitude balance | Hopfield >> others | More balanced |
| 7 | Force cosine similarity | ~0 (orthogonal) | > 0 (aligned) |
| 8 | Small-loop holonomy | 0 (flat) | > 0 (curved) |
| 9 | Activation pattern divergence | ~0 | > 0 |
| 10 | Subbundle entropy | Low | Higher |

### Next Steps (Phases 2-4)

- **Phase 2**: Test removing Forces 2 & 4 (causal conv & attention). If contextual manifold carries sufficient history, pure SDE may suffice.
- **Phase 3**: Non-Abelian gauge group ($SU(2)$), curvature-aware memory bank, self-synthesizing geometry, horizontal gradient projection.
- **Phase 4**: Scale comparison against standard Transformer baseline.

---

*See `README_PhD_Exploration_claude_v2.md` for the full mathematical autopsy, external paper connections, and all novel proposals.*